# proxy
> Caddy reverse proxy, CrowdSec security, swag containers and Cloudflare tunnel support

In [ ]:
#| default_exp proxy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path
from fastcore.all import L, store_attr, listify, joins

## Caddyfile builder

In [ ]:
#| export
class Caddyfile(L):
	'Fluent builder for production-ready Caddyfiles'
	def __init__(self, domain, app='app', port=5001):
		store_attr()
		super().__init__([])

	def _new(self, items):
		c = type(self)(self.domain, self.app, self.port)
		c.items = items
		return c

	def _add(self, *s): return self._new(self.items + list(s))
	def _has(self, tag): return any(t == tag for t, _ in self)
	def _get(self, tag): return L(self).filter(lambda x: x[0] == tag).itemgot(1)

	def email(self, addr):
		'ACME account email (global). Caddy works without it but uses anonymous account.'
		return self._add(('g', f'email {addr}'))

	def acme_dns(self, provider, token_env=None):
		'DNS-01 TLS challenge — needed when port 80 is unavailable (e.g. behind Cloudflare tunnel)'
		token = f'{{${token_env}}}' if token_env else ''
		return self._add(('s', f'\ttls {{\n\t\tacme_dns {provider} {token}\n\t}}'))

	def cloudflared(self):
		'Use http:// prefix — required when running behind cloudflared tunnel (no public port 80/443)'
		return self._add(('f', 'cloudflared'))

	def crowdsec(self, api_url='http://crowdsec:8080', key_env='CROWDSEC_API_KEY'):
		'CrowdSec bouncer — plugin, not built-in'
		return self._add(('s', f'\tcrowdsec {{\n\t\tapi_url {api_url}\n\t\tapi_key {{${key_env}}}\n\t}}'))

	def rate_limit(self, requests='100', window='1m'):
		'Rate limiting'
		return self._add(('s', f'\trate_limit {{\n\t\tzone dynamic {{\n\t\t\tkey {{http.request.remote_ip}}\n\t\t\tevents {requests}\n\t\t\twindow {window}\n\t\t}}\n\t}}'))

	def max_body(self, size='100m'):
		'Request body limit'
		return self._add(('s', f'\trequest_body {{\n\t\tmax_size {size}\n\t}}'))

	def spa(self, static_dir='/app/static'):
		'SPA preset: static files with fallback to index.html'
		return self._add(('spa', f'\troot * {static_dir}\n\ttry_files {{path}} /index.html\n\tfile_server'))

	def encode(self):
		'Compression — Caddy does NOT enable this by default, one word is enough'
		return self._add(('s', '\tencode'))

	def log(self, output='stderr'):
		'Access logging. Caddy logs errors by default; access logs are opt-in.'
		out = f'output {output}' if output != 'stderr' else ''
		return self._add(('s', f'\tlog {{\n\t\t{out}\n\t}}' if out else '\tlog'))

	def __str__(self):
		'Render full Caddyfile'
		parts = []
		g = list(self._get('g'))
		if g: parts.append('{\n\t' + '\n\t'.join(g) + '\n}')
		prefix = 'http://' if self._has('f') else ''
		site = [f'{prefix}{self.domain} {{'] + list(self._get('s'))
		if self._has('spa'): site += list(self._get('spa'))
		else: site.append(f'\treverse_proxy {self.app}:{self.port}')
		site.append('}')
		parts.append('\n'.join(site))
		return '\n'.join(parts)

	def __repr__(self): return str(self)

	def save(self, path='Caddyfile'):
		'Write to disk'
		Path(path).write_text(str(self))
		return self

## Caddyfile generation

`caddyfile()` generates the Caddyfile text. `caddy()` writes it and returns service kwargs for `Compose.svc()`.

In [ ]:
#| export
def caddy(domain,  # domain to serve (e.g. example.com or sub.example.com)
          app='app',  # upstream app name (must match Compose service name)
          port=5001,  # upstream app port
          email=None,  # ACME account email (optional but good practice)
          dns=None,  # DNS-01 provider name for TLS when port 80 is blocked (e.g. 'cloudflare' or 'duckdns')
          dns_token_env=None,  # env var name for DNS API token (default: {DNS}_API_TOKEN, e.g. CLOUDFLARE_API_TOKEN)
          crowdsec=False,  # enable CrowdSec bouncer plugin (not built-in, requires separate service). you don't need this if you're using the pre-built Caddy images with CrowdSec support.
          crowdsec_url='http://crowdsec:8080',  # CrowdSec API URL (default assumes separate 'crowdsec' service in same Compose network)
          cloudflared=False,  # prefix with http:// for Cloudflare tunnel setups (disables ACME, requires separate cloudflared service)
          encode=False,  # enable compression (not on by default)
          access_log=False  # enable access logging to stderr (errors only by default)
          )->Caddyfile:
	'Minimal Caddyfile for reverse-proxying app:port from domain. Optional ACME email, DNS-01 provider, CrowdSec protection, and Cloudflare tunnel support.'
	cf = Caddyfile(domain, app, port)
	if email: cf = cf.email(email)
	if dns and not cloudflared: cf = cf.acme_dns(dns, dns_token_env or f'{dns.upper()}_API_TOKEN')
	if cloudflared: cf = cf.cloudflared()
	if crowdsec: cf = cf.crowdsec(api_url=crowdsec_url)
	if encode: cf = cf.encode()
	if access_log: cf = cf.log()
	return cf

def caddy_api(domain,
              app='app',
              port=5001,
              email=None,
              dns=None,
              dns_token_env=None,
              cloudflared=False,
              crowdsec=False,
              crowdsec_url='http://crowdsec:8080',
              max_body='50m',
              rate='200',
              rate_window='1m'
              ) -> str:
	'Caddyfile preset for API services — adds rate limiting and body size cap'
	return caddy(domain, app, port, email=email, dns=dns, dns_token_env=dns_token_env,
	             cloudflared=cloudflared, crowdsec=crowdsec, crowdsec_url=crowdsec_url
	             ).rate_limit(rate, rate_window).max_body(max_body).encode()

In [ ]:
# Minimal
cf = str(caddy('myapp.example.com', port=5001))
assert 'myapp.example.com {' in cf
assert 'reverse_proxy app:5001' in cf
assert cf.count('{') == 1
print(cf)

myapp.example.com {
	reverse_proxy app:5001
}


In [ ]:
# With Cloudflare DNS
cf = str(caddy('myapp.example.com', port=5001, dns='cloudflare', email='me@example.com'))
assert 'email me@example.com' in cf
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
print(cf)

{
	email me@example.com
}
myapp.example.com {
	tls {
		acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}
	}
	reverse_proxy app:5001
}


In [ ]:
# With CrowdSec
cf = str(caddy('myapp.example.com', port=5001, crowdsec=True))
assert 'api_url http://crowdsec:8080' in cf
assert 'crowdsec' in cf
assert 'api_key {$CROWDSEC_API_KEY}' in cf
print(cf)

myapp.example.com {
	crowdsec {
		api_url http://crowdsec:8080
		api_key {$CROWDSEC_API_KEY}
	}
	reverse_proxy app:5001
}


In [ ]:
# Cloudflared mode: HTTP prefix
cf = str(caddy('myapp.example.com', port=5001, cloudflared=True))
assert cf.startswith('http://myapp.example.com {')
assert 'reverse_proxy app:5001' in cf
print(cf)

http://myapp.example.com {
	reverse_proxy app:5001
}


## Services

In [ ]:
#| export
def _caddy_img(crowdsec, cloudflare):
    if crowdsec and cloudflare: return 'ghcr.io/buildplan/csdp-caddy:latest'
    elif crowdsec: return 'serfriz/caddy-crowdsec:latest'
    elif cloudflare: return 'serfriz/caddy-cloudflare:latest'
    else: return 'caddy:2'

def caddy_svc(domain, app='app', port=5001, *,
              dns=None, email=None, crowdsec=False, cloudflared=False,
              conf='Caddyfile', **kw):
    'Write Caddyfile and return Caddy service kwargs for Compose.svc()'
    caddy(domain, app, port, email=email, dns=dns, cloudflared=cloudflared, crowdsec=crowdsec).save(conf)
    is_cf, tok = dns == 'cloudflare', f'{dns.upper()}_API_TOKEN' if dns else None
    env = dict() | {tok:'${%s}'%tok} if dns else {} | dict(CROWDSEC_API_KEY='${CROWDSEC_API_KEY}') if crowdsec else {}
    im, p = _caddy_img(crowdsec, is_cf), None if cloudflared else ['80:80', '443:443', '443:443/udp']
    conf_path = Path(conf)
    vol_key = str(conf_path.resolve()) if conf_path.is_absolute() else f'./{conf_path.name}'
    vol = {vol_key: '/etc/caddy/Caddyfile', 'caddy_data': '/data', 'caddy_config': '/config'}
    return dict(image=im, env=env or None, ports=p, volumes=vol, networks=['web'], depends_on=[app], restart='unless-stopped') | kw

In [ ]:
import tempfile
from fastops import Compose

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'caddy:2'
    assert kw['ports'] == ['80:80', '443:443', '443:443/udp']
    assert kw['depends_on'] == ['app']
    print('caddy_svc() basic OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', cloudflared=True, conf=f'{tmp}/Caddyfile')
    assert kw['ports'] is None
    assert kw['image'] == 'caddy:2'
    assert Path(f'{tmp}/Caddyfile').read_text().startswith('http://ex.com {')
    print('caddy_svc() cloudflared OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', crowdsec=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-crowdsec:latest'
    assert kw['env'] == {'CROWDSEC_API_KEY': '${CROWDSEC_API_KEY}'}
    print('caddy_svc() crowdsec OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', crowdsec=True, cloudflared=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-crowdsec:latest'
    assert kw['ports'] is None
    print('caddy_svc() crowdsec+cloudflared OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', dns='cloudflare', conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'serfriz/caddy-cloudflare:latest'
    assert kw['env'] == {'CLOUDFLARE_API_TOKEN': '${CLOUDFLARE_API_TOKEN}'}
    print('caddy_svc() cloudflare-dns OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.com', dns='cloudflare', crowdsec=True, conf=f'{tmp}/Caddyfile')
    assert kw['image'] == 'ghcr.io/buildplan/csdp-caddy:latest'
    print('caddy_svc() cloudflare-dns+crowdsec OK')

with tempfile.TemporaryDirectory() as tmp:
    kw = caddy_svc('ex.duckdns.org', dns='duckdns', conf=f'{tmp}/Caddyfile', image='serfriz/caddy-duckdns:latest')
    assert kw['image'] == 'serfriz/caddy-duckdns:latest'
    assert kw['env'] == {'DUCKDNS_API_TOKEN': '${DUCKDNS_API_TOKEN}'}
    cf = Path(f'{tmp}/Caddyfile').read_text()
    assert 'acme_dns duckdns {$DUCKDNS_API_TOKEN}' in cf
    print('caddy_svc() duckdns OK')

caddy_svc() basic OK
caddy_svc() cloudflared OK
caddy_svc() crowdsec OK
caddy_svc() crowdsec+cloudflared OK
caddy_svc() cloudflare-dns OK
caddy_svc() cloudflare-dns+crowdsec OK
caddy_svc() duckdns OK


In [ ]:
#| export
def cloudflared_svc(token_env='${CF_TUNNEL_TOKEN}', url=None, **kw):
    'Cloudflare tunnel service kwargs for Compose.svc(). Pass url to set ingress inline (e.g. url="http://caddy").'
    im = 'cloudflare/cloudflared:latest'
    cmd = f'tunnel --no-autoupdate run{" --url " + url if url else ""}'
    return dict(image=im, command=cmd, restart='unless-stopped', networks=['web'],
    env=dict(TUNNEL_TOKEN=token_env)) | kw

In [ ]:
kw = cloudflared_svc()
assert kw['image'] == 'cloudflare/cloudflared:latest'
assert kw['command'] == 'tunnel --no-autoupdate run'
assert kw['env'] == {'TUNNEL_TOKEN': '${CF_TUNNEL_TOKEN}'}
print('cloudflared_svc() OK')

kw2 = cloudflared_svc(url='http://caddy')
assert kw2['command'] == 'tunnel --no-autoupdate run --url http://caddy'
print('cloudflared_svc() with url OK')

cloudflared_svc() OK
cloudflared_svc() with url OK


In [ ]:
#| export
def crowdsec(collections=None, bouncer_key_env='CROWDSEC_BOUNCER_KEY', **kw):
    'CrowdSec agent service kwargs for Compose.svc()'
    cols = collections or ['crowdsecurity/linux', 'crowdsecurity/caddy', 'crowdsecurity/http-cve']
    cols, im = joins(' ',listify(cols)), 'crowdsecurity/crowdsec:latest'
    env = dict(COLLECTIONS=cols,BOUNCER_KEY_caddy=f'${{{bouncer_key_env}}}')
    vol = {'crowdsec-db': '/var/lib/crowdsec/data','crowdsec-config': '/etc/crowdsec'}
    return dict(image=im,env=env,volumes=vol, networks=['web'], restart='unless-stopped') | kw

In [ ]:
kw = crowdsec()
assert kw['image'] == 'crowdsecurity/crowdsec:latest'
assert 'crowdsecurity/caddy' in kw['env']['COLLECTIONS']
assert kw['env']['BOUNCER_KEY_caddy'] == '${CROWDSEC_BOUNCER_KEY}'
assert 'crowdsec-db' in kw['volumes']
print('crowdsec() OK')

kw2 = crowdsec(collections=['crowdsecurity/linux', 'crowdsecurity/nginx'])
assert 'crowdsecurity/nginx' in kw2['env']['COLLECTIONS']
print('crowdsec() custom collections OK')

crowdsec() OK
crowdsec() custom collections OK


## Example: FastHTML app with Caddy

Minimal stacks — run any with `dc.save('docker-compose.yml')` then `docker compose up -d`.

In [ ]:
tmp = tempfile.mkdtemp()

# Stack A: Direct (Caddy auto-TLS, ports 80+443 open)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, conf=f'{tmp}/Caddyfile'))
    .network('web', driver='bridge').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'caddy:2'
assert '80:80' in d['services']['caddy']['ports']
print('=== Stack A: Direct (Caddy auto-TLS) ===')
print(dc)

=== Stack A: Direct (Caddy auto-TLS) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: caddy:2
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpiwghosjd/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
networks:
  web:
    driver: bridge
volumes:
  caddy_data: null
  caddy_config: null



In [ ]:
tmp = tempfile.mkdtemp()
# Stack B: cloudflared tunnel (zero open ports)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, cloudflared=True, conf=f'{tmp}/Caddyfile'))
    .svc('cloudflared', **cloudflared_svc())
    .network('web').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert 'ports' not in d['services']['caddy']
assert d['services']['cloudflared']['image'] == 'cloudflare/cloudflared:latest'
print('=== Stack B: Cloudflared (zero open ports) ===')
print(dc)

=== Stack B: Cloudflared (zero open ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: caddy:2
    depends_on:
    - app
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp9285u1r5/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    restart: unless-stopped
    networks:
    - web
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null



In [ ]:
tmp = tempfile.mkdtemp()
# Stack C: CrowdSec + cloudflared (full security, zero open ports)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, crowdsec=True, cloudflared=True, conf=f'{tmp}/Caddyfile'))
    .svc('crowdsec', **crowdsec())
    .svc('cloudflared', **cloudflared_svc())
    .network('web')
    .volume('caddy_data').volume('caddy_config')
    .volume('crowdsec-db').volume('crowdsec-config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'serfriz/caddy-crowdsec:latest'
assert d['services']['crowdsec']['image'] == 'crowdsecurity/crowdsec:latest'
assert 'ports' not in d['services']['caddy']
print('=== Stack C: CrowdSec + cloudflared (zero open ports) ===')
print(dc)

=== Stack C: CrowdSec + cloudflared (zero open ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: serfriz/caddy-crowdsec:latest
    depends_on:
    - app
    environment:
    - CROWDSEC_API_KEY=${CROWDSEC_API_KEY}
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmprbikw4_g/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  crowdsec:
    image: crowdsecurity/crowdsec:latest
    environment:
    - COLLECTIONS=crowdsecurity/linux crowdsecurity/caddy crowdsecurity/http-cve
    - BOUNCER_KEY_caddy=${CROWDSEC_BOUNCER_KEY}
    volumes:
    - crowdsec-db:/var/lib/crowdsec/data
    - crowdsec-config:/etc/crowdsec
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    restart

In [ ]:
tmp = tempfile.mkdtemp()

# Stack D: Cloudflare DNS-01 (wildcard cert via ACME, ports 80+443 open, no tunnel)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001, dns='cloudflare', email='me@example.com', conf=f'{tmp}/Caddyfile'))
    .network('web').volume('caddy_data').volume('caddy_config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'serfriz/caddy-cloudflare:latest'
assert '80:80' in d['services']['caddy']['ports']
assert 'CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}' in d['services']['caddy']['environment']
cf = Path(f'{tmp}/Caddyfile').read_text()
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
print('=== Stack D: Cloudflare DNS-01 (wildcard cert, direct) ===')
print(dc)
print('--- Caddyfile ---')
print(cf)

=== Stack D: Cloudflare DNS-01 (wildcard cert, direct) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: serfriz/caddy-cloudflare:latest
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    environment:
    - CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmptftjy_x5/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null

--- Caddyfile ---
{
	email me@example.com
}
myapp.example.com {
	tls {
		acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}
	}
	reverse_proxy app:5001
}


In [ ]:
tmp = tempfile.mkdtemp()

# Stack E: Cloudflare DNS + CrowdSec (full security, direct ports open)
dc = (Compose()
    .svc('app', build='.', networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc('myapp.example.com', port=5001,
                              dns='cloudflare', crowdsec=True,
                              email='me@example.com', conf=f'{tmp}/Caddyfile'))
    .svc('crowdsec', **crowdsec())
    .network('web')
    .volume('caddy_data').volume('caddy_config')
    .volume('crowdsec-db').volume('crowdsec-config'))

d = dc.to_dict()
assert d['services']['caddy']['image'] == 'ghcr.io/buildplan/csdp-caddy:latest'
assert '80:80' in d['services']['caddy']['ports']
assert d['services']['crowdsec']['image'] == 'crowdsecurity/crowdsec:latest'
cf = Path(f'{tmp}/Caddyfile').read_text()
assert 'acme_dns cloudflare {$CLOUDFLARE_API_TOKEN}' in cf
assert 'crowdsec' in cf
print('=== Stack E: Cloudflare DNS + CrowdSec (full security, direct ports) ===')
print(dc)
print('--- Caddyfile ---')
print(cf)

=== Stack E: Cloudflare DNS + CrowdSec (full security, direct ports) ===
services:
  app:
    build: .
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: ghcr.io/buildplan/csdp-caddy:latest
    depends_on:
    - app
    ports:
    - 80:80
    - 443:443
    - 443:443/udp
    environment:
    - CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}
    volumes:
    - /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmparq8swvw/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  crowdsec:
    image: crowdsecurity/crowdsec:latest
    environment:
    - COLLECTIONS=crowdsecurity/linux crowdsecurity/caddy crowdsecurity/http-cve
    - BOUNCER_KEY_caddy=${CROWDSEC_BOUNCER_KEY}
    volumes:
    - crowdsec-db:/var/lib/crowdsec/data
    - crowdsec-config:/etc/crowdsec
    networks:
    - web
    restart: unless-stopped
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null
  crowd

## Integration test: FastHTML + Caddy + cloudflared

Live test — spins up a real stack and verifies the app is reachable over the internet.

**Prerequisites:**
- `DOMAIN` env var: hostname matching your Cloudflare tunnel ingress rule (e.g. `myapp.example.com`)
- `CF_TUNNEL_TOKEN` env var: from Cloudflare Zero Trust → Tunnels
- Tunnel ingress rule configured in Cloudflare dashboard: `DOMAIN → http://caddy`

In [ ]:
#| eval: false
import os, time, tempfile
from pathlib import Path
from fastcore.net import urlread
from fastops import Compose, Dockerfile, rmi

DOMAIN = 'fastops.angalama.com'           # e.g. myapp.example.com
# CF_TUNNEL_TOKEN read from host env by docker-compose: ${CF_TUNNEL_TOKEN}

# ── app ───────────────────────────────────────────────────────────────────────
app_dir = Path(tempfile.mkdtemp(dir='.'))
(app_dir / 'main.py').write_text('''\
from fasthtml.common import *
app, rt = fast_app()

@rt('/')
def get(): return Titled('fastops-proxy-test', P('Caddy + cloudflared ✓'))

serve(host='0.0.0.0', port=5001)
''')
(app_dir / 'requirements.txt').write_text('python-fasthtml\n')

tag = 'fastops-proxy-test:latest'
(Dockerfile()
    .from_('ghcr.io/astral-sh/uv', 'python3.13-bookworm-slim')
    .workdir('/app')
    .copy('requirements.txt', '.')
    .run('pip install --no-cache-dir -r requirements.txt')
    .copy('.', '.')
    .expose(5001)
    .cmd(['python', 'main.py'])
).build(tag=tag, path=str(app_dir), no_creds=True)
print(f'Built: {tag}')

# ── compose stack ─────────────────────────────────────────────────────────────
inf_dir = Path(tempfile.mkdtemp(dir='.'))
cf_path = str(inf_dir / 'Caddyfile')
dc_path = str(inf_dir / 'docker-compose.yml')

dc = (Compose()
    .svc('app', image=tag, networks=['web'], restart='unless-stopped')
    .svc('caddy', **caddy_svc(DOMAIN, dns='cloudflare', cloudflared=True, conf=cf_path))
    .svc('cloudflared', **cloudflared_svc(url='http://caddy'))
    .network('web').volume('caddy_data').volume('caddy_config'))

print(dc)
print('--- Caddyfile ---')
print(Path(cf_path).read_text())

# ── run & verify ──────────────────────────────────────────────────────────────
try:
	dc.up(path=dc_path)
	print('Waiting for cloudflared tunnel to connect...')
	html = None
	for i in range(12):
		time.sleep(10)
		try: html = urlread(f'https://{DOMAIN}'); break
		except Exception as e:
			print(dc.logs(path=dc_path))
			# print(f'  [{(i+1)*10}s] not ready: {e}')
	print('=== container logs ===')
	print(dc.logs(path=dc_path))
	assert html and 'fastops-proxy-test' in html, f'Unexpected response: {html[:200] if html else "no response"}'
	print(f'✓ App reachable at https://{DOMAIN}')
finally:
	dc.down(path=dc_path, v=True, remove_orphans=True)
	rmi(tag, force=True)
	print('Cleaned up.')

Built: fastops-proxy-test:latest
services:
  app:
    image: fastops-proxy-test:latest
    networks:
    - web
    restart: unless-stopped
  caddy:
    image: serfriz/caddy-cloudflare:latest
    depends_on:
    - app
    environment:
    - CLOUDFLARE_API_TOKEN=${CLOUDFLARE_API_TOKEN}
    volumes:
    - /Users/71293/code/personal/fastops/nbs/tmpwukinjbh/Caddyfile:/etc/caddy/Caddyfile
    - caddy_data:/data
    - caddy_config:/config
    networks:
    - web
    restart: unless-stopped
  cloudflared:
    image: cloudflare/cloudflared:latest
    command: tunnel --no-autoupdate run --url http://caddy
    environment:
    - TUNNEL_TOKEN=${CF_TUNNEL_TOKEN}
    restart: unless-stopped
    networks:
    - web
networks:
  web: null
volumes:
  caddy_data: null
  caddy_config: null

--- Caddyfile ---
http://fastops.angalama.com {
	reverse_proxy app:5001
}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()